In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)
import joblib  # Library untuk menyimpan model agar bisa dipakai di backend website
import os

# ===================================================
# 1. DATA PREPARATION & CLEANING (GLOBAL)
# ===================================================

# Membaca dataset
df = pd.read_csv('Rice_filtered_rice_commodities.csv')

# Membersihkan baris pertama jika berisi tag metadata (#date, #adm1+name, dll)
if str(df.iloc[0]['date']).startswith('#'):
    df_clean = df.iloc[1:].copy()
else:
    df_clean = df.copy()

# Konversi tipe data ke format yang benar
df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')
df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')

# Filter komoditas spesifik Beras (Rice) dan pastikan kolom admin1 tidak kosong
df_rice = df_clean[(df_clean['commodity'] == 'Rice') & (df_clean['admin1'].notna())].copy()

# Ambil daftar unik semua provinsi/daerah yang ada di dataset
daftar_daerah = df_rice['admin1'].unique()
print(f"--- Berhasil Memuat Data ---")
print(f"Total Daerah Terdeteksi: {len(daftar_daerah)} Daerah\n")


# Fungsi cetak evaluasi lengkap yang disesuaikan dari kode referensi Anda
def cetak_evaluasi_daerah(y_true, y_pred, nama_model, nama_daerah):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape_persen = mean_absolute_percentage_error(y_true, y_pred) * 100
    akurasi_model_persen = 100 - mape_persen

    print(f"==================================================")
    print(f" METRIK EVALUASI MODEL: {nama_model.upper()} - {nama_daerah}")
    print(f"==================================================")
    print(f"AKURASI MODEL PREDIKSI             : {akurasi_model_persen:.2f}%")
    print(f"--------------------------------------------------")
    print(f"1. MAPE (Persentase Eror Meleset)  : {mape_persen:.2f}%")
    print(f"2. MAE (Mean Absolute Error)       : Rp{mae:.2f}")
    print(f"3. RMSE (Root Mean Squared Error)  : Rp{rmse:.2f}")
    print(f"4. R2 Score (Koef. Determinasi)    : {r2:.4f}")
    print(f"==================================================\n")


# ===================================================
# 2. DYNAMIC PIPELINE PER DAERAH
# ===================================================

def train_dan_save_model(df_sumber, nama_daerah):
    """
    Fungsi otomatis untuk melakukan preprocessing, modeling, evaluasi, 
    dan penyimpanan model secara mandiri per wilayah.
    """
    # 1. Filter data berdasarkan wilayah pilihan
    df_daerah = df_sumber[df_sumber['admin1'] == nama_daerah].copy()
    
    # 2. Agregasi rata-rata harga per bulan khusus untuk wilayah tersebut
    df_monthly = df_daerah.groupby('date')['price'].mean().to_frame()
    
    if len(df_monthly) < 10:
        print(f"[Skip] Data untuk {nama_daerah} terlalu sedikit untuk dimodelkan.\n")
        return None

    # 3. Penyelarasan indeks bulanan penuh & Linear Interpolation per daerah
    tanggal_terakhir_data = df_monthly.index.max()
    all_months = pd.date_range(start=df_monthly.index.min(), end=tanggal_terakhir_data, freq='MS')
    all_months = all_months.map(lambda x: x.replace(day=15))
    df_monthly = df_monthly.reindex(all_months)
    
    # Mengisi kekosongan data dengan metode linier interpolasi khusus tren daerah tersebut
    df_monthly['price'] = df_monthly['price'].interpolate(method='linear')
    df_monthly = df_monthly.reset_index().rename(columns={'index': 'date'})
    
    # 4. Ekstraksi komponen waktu & lag sekuensial wilayah
    df_monthly['Bulan_Angka'] = df_monthly['date'].dt.month
    df_monthly['Tahun'] = df_monthly['date'].dt.year
    df_monthly['Lag_1'] = df_monthly['price'].shift(1)
    df_monthly['Lag_2'] = df_monthly['price'].shift(2)
    
    # Menghapus baris kosong akibat shifting lag
    df_ml = df_monthly.dropna().copy()
    
    # Menentukan Fitur (X) dan Target (y)
    X = df_ml[['Lag_1', 'Lag_2', 'Bulan_Angka', 'Tahun']]
    y = df_ml['price']
    
    # 5. Split Train & Test secara kronologis (70% Train, 30% Test)
    split_idx = int(len(df_ml) * 0.70)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    
    # 6. Modeling Menggunakan Linear Regression (Model terbaik dari riset Anda)
    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)
    y_pred_lr = model_lr.predict(X_test)
    
    # 7. Tampilkan hasil performa model untuk daerah ini
    cetak_evaluasi_daerah(y_test, y_pred_lr, "Linear Regression", nama_daerah)
    
    # 8. EXPORT MODEL (.pkl): Proses Krusial untuk Full-Stack Web Development
    # Menyimpan file model ke dalam folder agar bisa di-load oleh Flask / FastAPI
    #os.makedirs('saved_models', exist_ok=True)
    #nama_file = f"saved_models/model_rice_{nama_daerah.lower().replace(' ', '_')}.pkl"
    #joblib.dump(model_lr, nama_file)
    #print(f"[SUKSES] Model untuk {nama_daerah} berhasil diexport ke: {nama_file}\n")
    
    return model_lr

# ===================================================
# 3. RUNNING BATCH TRAINING (UNTUK SEMUA DAERAH)
# ===================================================

# Mengambil semua provinsi unik secara otomatis dari dataset
daerah_target = df_rice['admin1'].unique()

print(f"--- Memulai Proses Training untuk SEMUA Daerah ({len(daerah_target)} Wilayah) ---")

# List untuk mencatat daerah mana saja yang sukses atau gagal/di-skip
daerah_sukses = []
daerah_skip = []

for daerah in daerah_target:
    # Fungsi ini akan otomatis membuat, mengevaluasi, dan menyimpan model .pkl per daerah
    model = train_dan_save_model(df_rice, daerah)
    
    if model is not None:
        daerah_sukses.append(daerah)
    else:
        daerah_skip.append(daerah)

print("--------------------------------------------------")
# Rangkuman akhir proses training
print(f"PROSES SELESAI!")
print(f"Jumlah Daerah Berhasil Dibuatkan Model : {len(daerah_sukses)}")
print(f"Jumlah Daerah Di-skip (Data Kurang)    : {len(daerah_skip)}")
if len(daerah_skip) > 0:
    print(f"Daerah yang di-skip: {daerah_skip}")
print("--------------------------------------------------")

--- Berhasil Memuat Data ---
Total Daerah Terdeteksi: 31 Daerah

--- Memulai Proses Training untuk SEMUA Daerah (31 Wilayah) ---
 METRIK EVALUASI MODEL: LINEAR REGRESSION - ACEH
AKURASI MODEL PREDIKSI             : 96.99%
--------------------------------------------------
1. MAPE (Persentase Eror Meleset)  : 3.01%
2. MAE (Mean Absolute Error)       : Rp363.34
3. RMSE (Root Mean Squared Error)  : Rp509.85
4. R2 Score (Koef. Determinasi)    : 0.7633

 METRIK EVALUASI MODEL: LINEAR REGRESSION - BALI
AKURASI MODEL PREDIKSI             : 98.10%
--------------------------------------------------
1. MAPE (Persentase Eror Meleset)  : 1.90%
2. MAE (Mean Absolute Error)       : Rp234.07
3. RMSE (Root Mean Squared Error)  : Rp299.86
4. R2 Score (Koef. Determinasi)    : 0.9242

 METRIK EVALUASI MODEL: LINEAR REGRESSION - BANTEN
AKURASI MODEL PREDIKSI             : 97.77%
--------------------------------------------------
1. MAPE (Persentase Eror Meleset)  : 2.23%
2. MAE (Mean Absolute Error)      